## **Case 4: Caroline Bezengi oil spill, Hallaniyat Islands, Oman (June to September 2026)** <br/>

On 8 June 2026, an explosion disabled the Caroline Bezengi, a sanctioned tanker carrying close to a million barrels of Russian crude to India. The crew was taken off about four days later, and around 23 June the tanker ended up on the rocks at the southwest tip of Jazirat Al Qibliyyah, one of the Hallaniyat Islands off the Dhofar coast of Oman ([Maritime Executive](https://maritime-executive.com/article/stricken-tanker-caroline-bezengi-continues-to-leak-oil-off-oman))

It leaked through the Khareef monsoon, when strong winds and waves kept salvage crews away. On 10 August Oman put the slick at about 390 km² ([Reuters via gCaptain](https://gcaptain.com/oil-spill-off-oman-from-tanker-under-sanctions-against-russia-spreads-over-huge-area/)), and by 13 August oil had reached the beaches at Ras Madrakah, up to 40 km of coastline ([Al Jazeera](https://www.aljazeera.com/news/2026/8/13/oman-says-oil-spill-from-stricken-tanker-has-reached-its-coast))

This notebook looks at what NISAR saw: when the tanker shows up on the rocks, how the slick grows from one pass to the next, and where it drifts

**Main sources** <br/>

- [Challenge resources](https://www.spaceappschallenge.org/2026/challenges/dancing-with-the-sars/?tab=resources)

- [NASA Earthdata Search](https://search.earthdata.nasa.gov/search)

- [CMR search API](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html)

- [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/)

- [Earthdata Login](https://urs.earthdata.nasa.gov/)

- [NISAR available data](https://nisar-docs.asf.alaska.edu/availability-overview)

- [SEOS, oil on radar](https://seos-project.eu/marinepollution/marinepollution-c02-s02-p03.html)

- Event: [Maritime Executive, 23 Aug](https://maritime-executive.com/article/stricken-tanker-caroline-bezengi-continues-to-leak-oil-off-oman), [Maritime Executive, 9 Aug](https://maritime-executive.com/article/wrecked-tanker-continues-uncontrolled-oil-discharge-off-oman), [Reuters via gCaptain](https://gcaptain.com/oil-spill-off-oman-from-tanker-under-sanctions-against-russia-spreads-over-huge-area/), [Al Jazeera](https://www.aljazeera.com/news/2026/8/13/oman-says-oil-spill-from-stricken-tanker-has-reached-its-coast)

- [GCOV](https://nisar-docs.asf.alaska.edu/gcov)

- Places and coastline: [Hallaniyat Islands](https://en.wikipedia.org/wiki/Khuriya_Muriya_Islands), [OpenStreetMap](https://www.openstreetmap.org/), [Terrain Tiles](https://registry.opendata.aws/terrain-tiles/)

**Summary** <br/>

- NISAR saw the wreck about every 6 days from June to October, from two stacks: `150_D` (HH) and `056_A` (VV), with the grounding around 23 June bracketed by passes on Jun 18 and Jun 24. The area was never switched to Urgent Response

- Over open water coherence is close to zero, so the spill is told through backscatter: oil calms the sea and shows up darker than the water around it, here only about 1 to 2 dB darker at L-band on most passes

- Dark streaks show up north-east of the island from mid-July. The slick in view grows through August and is largest in September, a few hundred km² of the box, drifting north-east toward the coast as reported

- Thin streaks only about 1 dB darker, like on Aug 11, are visible by eye but fall below the threshold, and the calm, low-wind sea on Oct 10 is left out, since calm water looks dark too

- The overlays, places, mini charts and the manifest for the app are exported to `data/case4/export/`

<br/>

**Super Intelligence (SI) usage acknowledgement**

- **Tool used**: Claude (Anthropic)

- **Affected files**: `dataScience/notebooks/case4.ipynb` and its exports in `dataScience/data/case4/export/`

- **Purpose**: used to find and check the event sources, and to draft the code and write-ups for the data search, slick detection and export for the app

- **Human oversight**: all SI-generated code and text was reviewed, run and checked against the cited sources by the team, and adjusted to our NISAR data and the app's case study format


<br/>
<br/>

### 1. Data Retrieval (API Connection)

Find the NISAR data for this case through **[CMR (Common Metadata Repository)](https://www.earthdata.nasa.gov/about/esdis/eosdis/cmr)**, NASA's catalogue of every Earthdata dataset ([Earthdata Search](https://www.earthdata.nasa.gov/learn/earthdata-search) runs on it), then check if we can read the actual files

- Searching CMR is public, no login needed ([CMR search API](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html))

- The data files need an Earthdata login ([Accessing NISAR data](https://nisar-docs.asf.alaska.edu/access-overview))

#### 1.1 NISAR collections

- **Level 1 (in radar coordinates)**: radar geometry, not on a map yet

- **Level 2 (in geographic coordinates)**: put on a map grid and terrain corrected, with a quality mask

- **Level 3 (derived geophysical parameters)**: a physical quantity worked out from Level 2, e.g. soil moisture

([NISAR data products](https://nisar-docs.asf.alaska.edu/products-overview))

<br/>

Each dataset in CMR is a **collection** with a short name. A keyword search for "NISAR" lists them all ([keyword search](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html#c-keyword)), then keep the Level 2 and Level 3 ones

In [ ]:
# view NISAR collections
import requests
import pandas as pd
pd.set_option("display.max_colwidth", 80)

CMR = "https://cmr.earthdata.nasa.gov/search"

resp = requests.get(f"{CMR}/collections.json", params={"keyword": "NISAR", "page_size": 100}, timeout=60)
entries = resp.json()["feed"]["entry"]

collections = pd.DataFrame([(e["short_name"], e["title"], e["time_start"][:10]) for e in entries],
                           columns=["short_name", "title", "data_starts"])

# keep the level 2 and 3 science products, plus urgent response
collections = collections[collections.short_name.str.match(r"NISAR_(L2|L3|UR_L2)")].sort_values("short_name")
collections

- Each product comes in up to three quality stages: **Beta** (early, not for science), **Provisional** (calibrated, partly validated, from July 20, 2026) and **Validated** (fully calibrated and validated, released November 6, 2026) ([available data](https://nisar-docs.asf.alaska.edu/availability-overview))

- The spill runs from June to August 2026, so both **Provisional** and **Validated** are worth checking, compared in 1.2

<br/>

The products most likely to be useful for this case:

- **GCOV**: radar backscatter for each pass. Oil calms the small ripples on the sea surface, so less of the radar pulse bounces back and the slick shows up dark, while a ship shows up as a bright spot ([SEOS, oil on radar](https://seos-project.eu/marinepollution/marinepollution-c02-s02-p03.html))

- **GUNW**: coherence between two passes 12 days apart. Open water changes constantly, so it has almost no coherence, this is checked rather than assumed

- **Urgent Response**: passes processed fast after a disaster

([NISAR data products](https://nisar-docs.asf.alaska.edu/products-overview))

<br/>

#### 1.2 Granules over the study area

A **granule** is one file: one pass over one area for GCOV, or one pair of passes for GUNW

The search below asks CMR for every granule of a collection that touches our box within a date range

- `bounding_box`: `west,south,east,north` in degrees

- `temporal`: start and end date

- `.umm_json`: returns the full record for each granule, including the download links

([granule search](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html#granule-search-by-parameters))

<br/>

Study area: from the wreck at Jazirat Al Qibliyyah and the rest of the Hallaniyat Islands, north-east along the coast to Ras Madrakah, where the oil came ashore, drawn on [bboxfinder](http://bboxfinder.com/#17.20,55.80,19.20,58.00). Window: May to October 2026, from before the explosion on 8 June to well after the oil reached the coast

Place positions are from OpenStreetMap ([OpenStreetMap](https://www.openstreetmap.org/)) and Wikipedia for the islands ([Hallaniyat Islands](https://en.wikipedia.org/wiki/Khuriya_Muriya_Islands))

In [ ]:
import plotly.graph_objects as go

AOI = (55.80, 17.20, 58.00, 19.20)   # Area of Interest: west, south, east, north
west, south, east, north = AOI

# places to label on the map
places = pd.DataFrame([
    ("Jazirat Al Qibliyyah (wreck)", 56.3375, 17.5000),
    ("Al Hallaniyah", 56.0153, 17.5168),
    ("Shuwaymiyah", 55.6097, 17.8853),
    ("Sharbithat", 56.2721, 17.9385),
    ("Ras Madrakah", 57.7824, 18.9806),
], columns=["name", "lon", "lat"])

# satellite basemap with the study area and places drawn on top
fig = go.Figure(go.Scattermap(lon=[west, east, east, west, west], lat=[south, south, north, north, south],
                              mode="lines", line=dict(color="red", width=2), name="study area"))
fig.add_trace(go.Scattermap(lon=places.lon, lat=places.lat, mode="markers+text", text=places.name, textposition="top right",
                            marker=dict(size=9, color="yellow"), textfont=dict(color="white"), name="places"))
fig.update_layout(
    map=dict(style="white-bg", center=dict(lon=(west + east) / 2, lat=(south + north) / 2), zoom=7,
             layers=[dict(below="traces", sourcetype="raster", sourceattribution="Esri World Imagery",
                          source=["https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"])]),
    height=500, margin=dict(l=0, r=0, t=0, b=0), showlegend=False)
fig.show()

In [ ]:
START, END = "2026-05-01", "2026-10-31"

# function to find every NISAR file over the study area within the date window
def search_granules(short_name, bbox=AOI, start=START, end=END):
    """
    Search CMR for every granule of one collection over a box and date range

    Args:
        short_name: Collection short name, e.g. "NISAR_L2_GCOV_PROVISIONAL_V1"
        bbox: Bounding box (west, south, east, north) in degrees
        start: First date to include, "YYYY-MM-DD"
        end: Last date to include, "YYYY-MM-DD"

    Returns:
        DataFrame with one row per granule: name, start and end time, size in GB, and the .h5 download link
    """
    rows, page = [], 1
    while True:
        params = {"short_name": short_name, "bounding_box": ",".join(map(str, bbox)),
                  "temporal": f"{start}T00:00:00Z,{end}T23:59:59Z", "page_size": 200, "page_num": page}
        items = requests.get(f"{CMR}/granules.umm_json", params=params, timeout=120).json()["items"]
        for item in items:
            umm = item["umm"]
            urls = [u["URL"] for u in umm["RelatedUrls"] if u["URL"].startswith("https")]
            rows.append({
                "granule": umm["GranuleUR"],
                "start": pd.to_datetime(umm["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]),
                "end": pd.to_datetime(umm["TemporalExtent"]["RangeDateTime"]["EndingDateTime"]),
                "size_gb": round(umm["DataGranule"]["ArchiveAndDistributionInformation"][0]["SizeInBytes"] / 1e9, 2),
                "url": next(u for u in urls if u.endswith(".h5")),
            })
        # CMR returns at most 200 per page, so a short page means we have them all
        if len(items) < 200:
            break
        page += 1
    return pd.DataFrame(rows, columns=["granule", "start", "end", "size_gb", "url"])

In [54]:
# search each collection over the study area for May to October
gcov = search_granules("NISAR_L2_GCOV_PROVISIONAL_V1")
gcov_validated = search_granules("NISAR_L2_GCOV_V1")
gunw = search_granules("NISAR_L2_GUNW_PROVISIONAL_V1")
urgent = search_granules("NISAR_UR_L2")

# how many files each one has, and how much it would be to download them all
found = {"GCOV Provisional": gcov, "GCOV Validated": gcov_validated, "GUNW Provisional": gunw, "Urgent Response": urgent}
pd.DataFrame({"granules": [len(d) for d in found.values()],
              "total_gb": [round(d.size_gb.sum(), 1) for d in found.values()],
              "first": [d.start.min() for d in found.values()],
              "last": [d.start.max() for d in found.values()]}, index=list(found))

,granules,total_gb,first,last
GCOV Provisional,91,103.3,2026-06-17 14:40:56+00:00,2026-10-21 01:33:24+00:00
GCOV Validated,6,10.7,2026-10-22 14:49:12+00:00,2026-10-28 01:41:24+00:00
GUNW Provisional,57,99.9,2026-06-17 14:40:56+00:00,2026-10-09 01:33:24+00:00
Urgent Response,0,0.0,NaT,NaT


- **GCOV Provisional**: 91 passes from 17 June to 21 October, about 103 GB. Five different stacks cross the box, so there's a pass almost every one to six days, apart from the instrument gap from Jul 27 to Aug 10 ([available data](https://nisar-docs.asf.alaska.edu/availability-overview))

- **GCOV Validated**: only from 22 October over this box, after the spill, so **Provisional** is the one to use

- **GUNW**: 57 pairs, about 100 GB

- **Urgent Response**: none over the box

**Urgent Response**

No Urgent Response granules over the box, so check a much wider area in case the processing was set up around the coast instead

In [55]:
# urgent response over a much wider box, the whole Arabian Sea coast of Oman
urgent_wide = search_granules("NISAR_UR_L2", bbox=(52.0, 15.0, 60.0, 23.0))
print(f"urgent response granules over the wider box: {len(urgent_wide)}")
urgent_wide[["start", "granule"]]

urgent response granules over the wider box: 0


,start,granule


- None anywhere along the coast either. Unlike case 1, NISAR never switched this area to emergency processing, the spill was only picked up by the regular passes

- That also makes sense for an oil spill: NISAR mainly images land and sea ice ([About NISAR](https://nisar-docs.asf.alaska.edu/nisar-intro)), the sea around the islands is only covered because it sits next to the coast

<br/>

#### 1.3 Accessing the files

**Earthdata login**

The `.h5` files need an Earthdata Login token, saved in `dataScience/.env` as `EDL_TOKEN=...` ([how to get a token](https://urs.earthdata.nasa.gov/documentation/for_users/user_token))

In [56]:
from pathlib import Path

# read EDL_TOKEN from dataScience/.env
env = dict(line.split("=", 1) for line in Path("../.env").read_text().splitlines() if "=" in line and not line.startswith("#"))
AUTH = {"Authorization": f"Bearer {env['EDL_TOKEN'].strip()}"}

Check the token works by asking for just the first 8 bytes of one real file. Every HDF5 file starts with the same signature, `\x89HDF\r\n\x1a\n`

- `206` + that signature: the token works

- `200` with a web page: the request got sent to the login page, so the token is missing or expired

In [57]:
# check if the first GCOV file is accessible with the Earthdata login token
url = gcov.url.iloc[0]
check = requests.get(url, headers={**AUTH, "Range": "bytes=0-7"}, timeout=60)
check.status_code, check.content[:8]

(206, b'\x89HDF\r\n\x1a\n')

<br/>

**Retrieving a file**

The guide mentions files can be streamed instead of downloaded ([Accessing NISAR data](https://nisar-docs.asf.alaska.edu/access-overview)). The download link redirects to a temporary signed link to the file in the cloud, and `h5py` can open it as a file-like object through `fsspec` ([h5py](https://docs.h5py.org/en/stable/high/file.html#python-file-like-objects)), so only the parts we actually read come over the network

In [58]:
import h5py
import fsspec

# function to open a NISAR file in the cloud without downloading the whole thing
def open_h5(url):
    """
    Open a NISAR .h5 file remotely, reading only the parts that get used

    Args:
        url: The .h5 download link from search_granules

    Returns:
        h5py File that can be read like a local file
    """
    # the download link redirects to a temporary signed link, grab it instead of following it
    signed = requests.get(url, headers=AUTH, allow_redirects=False, timeout=60).headers["Location"]

    # fsspec fetches the file in 4 MB pieces, only when h5py asks for them
    return h5py.File(fsspec.open(signed, "rb", block_size=4 * 2**20).open(), "r")

In [59]:
# quick test: list the top-level groups of one GCOV file
with open_h5(gcov.url.iloc[0]) as f:
    print(list(f.keys()))
    print(list(f["science/LSAR"].keys()))

['science']
['GCOV', 'identification']


<br/>

**What each file holds**

Each file can hold two **frequencies**: frequency A, the main one, and frequency B, a narrower band mainly collected for atmospheric corrections. Frequency B has bigger pixels, so its files are smaller and quicker to read, which suits a quick look over a large area ([About NISAR](https://nisar-docs.asf.alaska.edu/nisar-intro), [GCOV](https://nisar-docs.asf.alaska.edu/gcov))

In [ ]:
# for one pass of each frame and mode: which frequencies it holds, their polarizations and pixel size
frames = gcov.assign(frame=gcov.granule.str.extract(r"GCOV_\d{3}_(\d{3}_[AD]_\d{3}_\d{4}_[A-Z]{4})")[0]).drop_duplicates("frame")
rows = []
for r in frames.itertuples():
    with open_h5(r.url) as f:
        for freq, grid in f["science/LSAR/GCOV/grids"].items():
            x = grid["xCoordinates"]
            rows.append((r.frame, freq, [p.decode() for p in grid["listOfPolarizations"][()]],
                         float(x[1] - x[0]), x.shape[0], int(grid["projection"][()])))
pd.DataFrame(rows, columns=["frame", "frequency", "polarizations", "pixel_m", "width_px", "epsg"])

- **Frequency A**: 20 m pixels in most frames (10 m for `157_A_010`), with HH and HV, or HH only for the `SHSH` passes

- **Frequency B**: 80 m pixels in **every** frame, HH (and HV) for most, and VV only for the `0005 NASV` frames, which have no frequency A at all

- Every frame is on the same map projection, **EPSG 32640** (WGS 84 / UTM zone 40N, [epsg.io](https://epsg.io/32640))

- The slick covers hundreds of km², so the 80 m frequency B layer is the one every pass has in common, and much quicker to read over the whole box. The 20 m layer is for a closer look at the wreck

We can now find every NISAR file over the spill and read them remotely. Next is section 2: lining up the passes from the different stacks and checking how consistent they are

<br/>
<br/>

### 2. Exploratory Data Analysis

Before looking at the spill, check what's inside the files, how the passes from the different stacks line up, and how much of the box each one covers

#### 2.1 Inside a GCOV file

Each `.h5` file is **HDF5**, organised like files on a computer: **groups** work like folders, and the **datasets** inside them hold the actual data, as arrays ([data format](https://nisar-docs.asf.alaska.edu/data-format)). Frequency B is the layer every pass has (1.3), under `/science/LSAR/GCOV/grids/frequencyB/` ([GCOV](https://nisar-docs.asf.alaska.edu/gcov))

The file used here is the first pass of the frame the wreck sits in, `150_D_080`

In [ ]:
# view every dataset in the frequency B grid of one file over the wreck: name, shape and data type
wreck_file = gcov[gcov.granule.str.contains("_150_D_080_")].iloc[0]
with open_h5(wreck_file.url) as f:
    grid = f["science/LSAR/GCOV/grids/frequencyB"]
    for name, item in grid.items():
        if isinstance(item, h5py.Dataset):
            print(f"{name:35} {str(item.shape):20} {item.dtype}")

Useful:

- `HHHH`, `HVHV`: backscatter power for each polarization

- `mask`: a quality flag for every pixel

- `xCoordinates`, `yCoordinates`, `projection`: where each pixel sits on the map

The frequency B grid is 4,194 x 4,275 pixels at 80 m, about 335 x 342 km, the whole frame. Our box is about 230 x 220 km, so a good part of each frame

<br/>

Definition of terms:

- **Backscatter**: the radar sends out a pulse and measures how much of it bounces back. Rough surfaces send more back (bright), smooth ones bounce it away (dark). On the sea, wind makes small ripples that send some back, oil smooths them out, so a slick is darker than the water around it ([SEOS, oil on radar](https://seos-project.eu/marinepollution/marinepollution-c02-s02-p03.html))

- **Polarization**: the orientation of the radar wave sent and received, H (horizontal) or V (vertical). `HHHH` is sent H and received H, `VVVV` sent V and received V ([Intro to SAR](https://www.earthdata.nasa.gov/learn/earth-observation-data-basics/sar))

In [ ]:
# view the pass details stored in the file: track, frame, direction, dates and so on
with open_h5(wreck_file.url) as f:
    ident = f["science/LSAR/identification"]
    for name, item in ident.items():
        if isinstance(item, h5py.Dataset) and item.shape == () and name != "boundingPolygon":
            value = item[()]
            print(f"{name:40} {value.decode() if isinstance(value, bytes) else value}")

- `trackNumber` **150**, `frameNumber` **80**, `orbitPassDirection` **Descending**: the same details as in the granule name (2.2)

- `isUrgentObservation` **False**: a regular pass, not one ordered for an emergency, which fits 1.2

- `isFullFrame` **False**: this frame is only partly filled, so a pass over the box is split across frames `079` and `080`

- `processingDateTime`: this June pass was processed on July 21, the day after the Provisional release ([available data](https://nisar-docs.asf.alaska.edu/availability-overview))

<br/>

#### 2.2 Passes and consistency

The granule name holds the details of each pass ([naming conventions](https://nisar-docs.asf.alaska.edu/naming-conventions)):

- **track** and **direction** (A ascending / D descending): together the **stack**, which sees the area from the same angle every 12 days

- **frame**: which piece of the track. A pass over a big box can be split over two frames, e.g. `079` and `080`

- **mode**: the radar bandwidth, e.g. `2005` (20 + 5 MHz) or `0005` (5 MHz only), which sets the pixel size

- **pols**: the polarizations collected, e.g. `DHDH` (HH and HV), `SHSH` (HH only), `NASV` (VV only)

- **crid**: the processing version

In [ ]:
# split each GCOV granule name into its parts
NAME = (r"GCOV_(?P<cycle>\d{3})_(?P<track>\d{3})_(?P<direction>[AD])_(?P<frame>\d{3})_(?P<mode>\d{4})_(?P<pols>[A-Z]{4})"
        r"_\w_(?P<date>\d{8})T\d{6}_\d{8}T\d{6}_(?P<crid>[A-Z]\d{5})")

pieces = gcov.join(gcov.granule.str.extract(NAME))
pieces["date"] = pd.to_datetime(pieces.date)
pieces["stack_id"] = pieces.track + "_" + pieces.direction
pd.crosstab([pieces.stack_id, pieces.frame], [pieces["mode"], pieces.pols])

- **Five stacks** cross the box. Each pass is split over two frames, except `049_D`, which only has `079`

- `150_D` and `049_D` switch between `DHDH` (HH and HV) and `SHSH` (HH only) from one pass to the next, so **HH** is the layer every one of their passes has

- `056_A_010` and `128_A_010` only ever collect `0005 NASV`: 5 MHz, VV only, frequency B only (1.3)

In [ ]:
# pieces that show up more than once (same stack, frame, mode and date)
dupes = pieces[pieces.duplicated(["stack_id", "frame", "mode", "date"], keep=False)].sort_values(["stack_id", "date"])
dupes[["date", "stack_id", "frame", "mode", "pols", "crid", "size_gb"]]

The `157_A` pass on Sep 5 is listed twice with the same CRID, two copies of the same file in CMR. Keep one

In [ ]:
# keep one row per piece, the latest processing version, then count the passes (stack and date) and pieces per pass
pieces = (pieces.sort_values("crid", key=lambda c: c.str[1:].astype(int))
                .drop_duplicates(["stack_id", "frame", "mode", "date"], keep="last")
                .sort_values("start").reset_index(drop=True))
per_pass = pieces.groupby(["stack_id", "date"]).agg(pieces=("granule", "size"), frames=("frame", lambda f: " + ".join(sorted(set(f)))),
                                                   modes=("pols", lambda p: " + ".join(sorted(set(p)))))
per_pass.groupby("stack_id").agg(passes=("pieces", "size"), frames=("frames", "first"), modes=("modes", lambda m: sorted(set(m))))

9 or 10 passes per stack between June and October, one every 12 days, with the gap from Jul 27 to Aug 10

<br/>

**GUNW pairs**

A GUNW name holds two dates: the **reference** (earlier) and **secondary** (later) pass. Pairs should be 12 days apart ([GUNW](https://nisar-docs.asf.alaska.edu/gunw))

In [ ]:
# split each GUNW granule name into its parts, then keep the latest version of each pair
PAIR = (r"GUNW_\d{3}_(?P<track>\d{3})_(?P<direction>[AD])_(?P<frame>\d{3})_\d{3}_(?P<mode>\d{4})_(?P<pols>[A-Z]{2})"
        r"_(?P<reference>\d{8})T\d{6}_\d{8}T\d{6}_(?P<secondary>\d{8})T\d{6}_\d{8}T\d{6}_(?P<crid>[A-Z]\d{5})")

pairs = gunw.join(gunw.granule.str.extract(PAIR))
pairs["reference"], pairs["secondary"] = pd.to_datetime(pairs.reference), pd.to_datetime(pairs.secondary)
pairs["stack_id"] = pairs.track + "_" + pairs.direction
pairs["days"] = (pairs.secondary - pairs.reference).dt.days
pairs = (pairs.sort_values("crid", key=lambda c: c.str[1:].astype(int))
              .drop_duplicates(["stack_id", "frame", "reference", "secondary"], keep="last")
              .sort_values("reference").reset_index(drop=True))
pd.crosstab([pairs.stack_id, pairs.frame], pairs.days)

- Most pairs are 12 days apart, but frame `150_D_080`, the one with the wreck, has no 12-day pairs at all, only 24, 48 and 72 days

- Over open water coherence is close to zero anyway, the sea surface changes from one second to the next, so GUNW isn't used further. The spill is told through **backscatter** (GCOV)

<br/>

#### 2.3 Coverage of the study area

A pass can clip the edge of the box. Each granule's record includes the outline of the area it covers, so check which passes cover the wreck and how much of the box each one sees

In [ ]:
from shapely.geometry import Point, Polygon, box
from shapely.ops import unary_union

# function to get the outline of the area one granule covers
def footprint(short_name, granule):
    """
    Get a granule's footprint from CMR

    Args:
        short_name: Collection short name
        granule: Granule name

    Returns:
        shapely Polygon in longitude and latitude
    """
    params = {"short_name": short_name, "granule_ur": granule}
    umm = requests.get(f"{CMR}/granules.umm_json", params=params, timeout=60).json()["items"][0]["umm"]
    points = umm["SpatialExtent"]["HorizontalSpatialDomain"]["Geometry"]["GPolygons"][0]["Boundary"]["Points"]
    return Polygon([(p["Longitude"], p["Latitude"]) for p in points])

In [ ]:
# footprint of every piece, joined into one outline per pass
pieces["footprint"] = [footprint("NISAR_L2_GCOV_PROVISIONAL_V1", g) for g in pieces.granule]
passes = pieces.groupby(["stack_id", "date"]).agg(start=("start", "min"), footprint=("footprint", unary_union),
                                                  pols=("pols", lambda p: " + ".join(sorted(set(p))))).reset_index()

wreck = Point(56.3375, 17.50)   # Jazirat Al Qibliyyah
passes["covers_wreck"] = passes.footprint.apply(lambda poly: poly.contains(wreck))
passes["box_share"] = passes.footprint.apply(lambda poly: poly.intersection(box(*AOI)).area / box(*AOI).area).round(2)
passes.groupby("stack_id").agg(passes=("date", "size"), covers_wreck=("covers_wreck", "mean"), box_share=("box_share", "mean"))

- Only `150_D` covers the wreck on every pass, and `056_A` on 8 of its 10. They also see most of the box: 71% and 85% on average

- The other three stacks only clip the edge of the box (8 to 12%)

- So the analysis uses `150_D` and `056_A`. Their passes fall about 6 days apart (e.g. Jun 18, Jun 24, Jun 30, Jul 6), so together they see the wreck roughly every 6 days

In [ ]:
# one pass outline per stack on the map, with the study area in red and the wreck in yellow
fig = go.Figure()
colors = ["royalblue", "orange", "limegreen", "magenta", "cyan"]
for color, (stack_id, row) in zip(colors, passes.groupby("stack_id").first().iterrows()):
    polys = row.footprint.geoms if row.footprint.geom_type == "MultiPolygon" else [row.footprint]
    for i, poly in enumerate(polys):
        lon, lat = poly.exterior.xy
        fig.add_trace(go.Scattermap(lon=list(lon), lat=list(lat), mode="lines", line=dict(color=color, width=2),
                                    name=stack_id, showlegend=i == 0))
fig.add_trace(go.Scattermap(lon=[west, east, east, west, west], lat=[south, south, north, north, south],
                            mode="lines", line=dict(color="red", width=2), name="study area"))
fig.add_trace(go.Scattermap(lon=[wreck.x], lat=[wreck.y], mode="markers", marker=dict(size=10, color="yellow"), name="wreck"))
fig.update_layout(
    map=dict(style="white-bg", center=dict(lon=(west + east) / 2, lat=(south + north) / 2), zoom=5.5,
             layers=[dict(below="traces", sourcetype="raster", sourceattribution="Esri World Imagery",
                          source=["https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"])]),
    height=500, margin=dict(l=0, r=0, t=0, b=0),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255, 255, 255, 0.8)"))
fig.show()

Each outline is one pass per stack, in a different colour per stack, with the study area in red and the wreck in yellow. `150_D` and `056_A` run across the islands and the coast, the other three only touch the corners

<br/>

#### 2.4 Valid pixels and a first look over the sea

The `mask` layer flags each pixel ([GCOV](https://nisar-docs.asf.alaska.edu/gcov)):

- **1 to 5**: valid

- **0**: invalid

- **255**: outside the image

<br/>

Read the frequency B mask and HH (or VV) over the box for one pass from each stack that covers the wreck, to see how much of the box is usable and what the sea looks like

In [ ]:
import numpy as np
from pyproj import Transformer

# function to find the rows and columns of a grid that fall inside the study area
def aoi_window(grid, bbox=AOI):
    """
    Convert the study area box into a row and column window of a NISAR grid

    Args:
        grid: h5py group holding xCoordinates, yCoordinates and projection
        bbox: Bounding box (west, south, east, north) in degrees

    Returns:
        (rows, cols) slices for reading only the box from each layer, or None if the grid misses the box
    """
    to_grid = Transformer.from_crs(4326, int(grid["projection"][()]), always_xy=True)
    xs, ys = to_grid.transform([bbox[0], bbox[2], bbox[0], bbox[2]], [bbox[1], bbox[1], bbox[3], bbox[3]])
    x, y = grid["xCoordinates"][:], grid["yCoordinates"][:]
    cols = np.where((x >= min(xs)) & (x <= max(xs)))[0]
    rows = np.where((y >= min(ys)) & (y <= max(ys)))[0]
    if len(cols) == 0 or len(rows) == 0:
        return None
    return slice(rows[0], rows[-1] + 1), slice(cols[0], cols[-1] + 1)

In [ ]:
# function to read one frequency B layer over the box for one piece, with its coordinates
def read_box(url, layer):
    """
    Read a frequency B layer and the mask over the study area

    Args:
        url: The .h5 download link
        layer: "HHHH", "HVHV" or "VVVV", or "mask" for the mask only

    Returns:
        (data, mask, x, y), or None if the piece doesn't reach the box
    """
    with open_h5(url) as f:
        grid = f["science/LSAR/GCOV/grids/frequencyB"]
        window = aoi_window(grid)
        if window is None or layer not in grid:
            return None
        rows, cols = window
        return grid[layer][rows, cols], grid["mask"][rows, cols], grid["xCoordinates"][cols], grid["yCoordinates"][rows]

In [ ]:
# mask values over the box for the wreck frame of each stack, first pass after the instrument gap
first_after_gap = pieces[pieces.date >= "2026-08-10"].sort_values("date").groupby("stack_id").first()
rows = []
for stack_id, piece in first_after_gap.iterrows():
    result = read_box(piece.url, "mask")
    if result is None:
        continue
    values, counts = np.unique(result[0], return_counts=True)
    rows += [(stack_id, f"{piece.date:%b %d}", piece.frame, v, round(c / counts.sum(), 3)) for v, c in zip(values, counts)]
pd.DataFrame(rows, columns=["stack_id", "date", "frame", "mask value", "share"]).pivot_table(
    index=["stack_id", "date", "frame"], columns="mask value", values="share", fill_value=0)

- The share of `255` is the part of the box outside that one frame, the pass's other frame covers most of it

- Invalid pixels (`0`) are only 0.4 to 2% of the box, so the sea is valid wherever a frame reaches it

In [ ]:
import matplotlib.pyplot as plt

# HH (or VV) in dB over the box, one pass per stack, every frame of that pass placed by its coordinates
fig, axes = plt.subplots(1, len(first_after_gap), figsize=(5 * len(first_after_gap), 5))
to_utm = Transformer.from_crs(4326, 32640, always_xy=True)
(bx0, by0), (bx1, by1) = to_utm.transform(west, south), to_utm.transform(east, north)
for ax, (stack_id, piece) in zip(np.atleast_1d(axes), first_after_gap.iterrows()):
    for p in pieces[(pieces.stack_id == stack_id) & (pieces.date == piece.date)].itertuples():
        result = read_box(p.url, "HHHH") or read_box(p.url, "VVVV")
        if result is None:
            continue
        data, mask, x, y = result
        db = np.where((mask >= 1) & (mask <= 5) & (data > 0), 10 * np.log10(np.where(data > 0, data, 1)), np.nan)
        ax.imshow(db, extent=[x[0], x[-1], y[-1], y[0]], cmap="gray", vmin=-30, vmax=0)
    ax.plot(*to_utm.transform(wreck.x, wreck.y), "y+", markersize=12)
    ax.set_xlim(bx0, bx1)
    ax.set_ylim(by0, by1)
    ax.set_title(f"{stack_id}, {piece.date:%b %d}")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

**Reading the first look** (the yellow cross is the wreck)

- Land at the top is bright and textured, the sea is darker and smooth, the Hallaniyat Islands are the bright shapes at the bottom left

- `150_D`, **Aug 11**, the first pass after the instrument gap: a dark trail runs east and north-east from the wreck for tens of km. That's the slick, oil calming the ripples on the water

- `056_A`, **Aug 17**, in VV six days later: the same dark streaks north-east of the wreck

- `056_A`'s two frames leave an empty band across the middle of the box. The faint straight lines across the sea in `150_D` follow the satellite's track, not anything on the water, so they come from the radar rather than oil

- The other three stacks barely reach the box

<br/>

Section 2 leaves us with one row per pass, and two stacks, `150_D` and `056_A`, that see the wreck about every 6 days. The slick is already visible in the first look. Next is section 3: lining the passes up with the timeline of the spill and measuring the slick on each one

<br/>
<br/>

### 3. Case Study Analysis

Line the passes up with what happened to the tanker, look for the tanker itself on the rocks, then find the slick on every pass and measure it

#### 3.1 Temporal analysis

Key dates, from the news reports in the introduction:

- **8 June**: explosion on board, the tanker is disabled ([Maritime Executive](https://maritime-executive.com/article/stricken-tanker-caroline-bezengi-continues-to-leak-oil-off-oman))

- **About 23 June**: the tanker ends up on the rocks at the southwest tip of Jazirat Al Qibliyyah (same source)

- **10 August**: Oman puts the slick at about 390 km² ([Reuters via gCaptain](https://gcaptain.com/oil-spill-off-oman-from-tanker-under-sanctions-against-russia-spreads-over-huge-area/))

- **13 August**: oil reaches the beaches at Ras Madrakah ([Al Jazeera](https://www.aljazeera.com/news/2026/8/13/oman-says-oil-spill-from-stricken-tanker-has-reached-its-coast))

In [ ]:
KEY_DATES = {
    "explosion": pd.Timestamp("2026-06-08"),
    "aground": pd.Timestamp("2026-06-23"),
    "slick 390 km²": pd.Timestamp("2026-08-10"),
    "oil on the coast": pd.Timestamp("2026-08-13"),
}
GAP = (pd.Timestamp("2026-07-27"), pd.Timestamp("2026-08-10"))

# every pass per stack against the key dates, filled where the pass covers the wreck
fig = go.Figure()
for stack_id, group in passes.groupby("stack_id"):
    fig.add_trace(go.Scatter(x=group.date, y=[stack_id] * len(group), mode="markers", name=stack_id,
                             marker=dict(size=10, symbol=["circle" if c else "circle-open" for c in group.covers_wreck]),
                             text=group.pols, hovertemplate="%{x|%b %d}<br>%{text}"))
fig.add_vrect(x0=GAP[0], x1=GAP[1], fillcolor="gray", opacity=0.15, line_width=0)
# labels on two heights, so dates close together don't overlap
for i, (label, date) in enumerate(KEY_DATES.items()):
    fig.add_vline(x=date, line=dict(color="red", dash="dash"))
    fig.add_annotation(x=date, y=1.07 + 0.13 * (i % 2), yref="paper", text=label, showarrow=False, font=dict(color="red", size=11))
fig.update_layout(height=300, margin=dict(l=0, r=0, t=60, b=0), showlegend=False)
fig.show()

In [ ]:
# passes that see the wreck, with how long after it went aground
wreck_passes = passes[passes.covers_wreck & passes.stack_id.isin(["150_D", "056_A"])].sort_values("date").reset_index(drop=True)
wreck_passes["days_since_aground"] = (wreck_passes.date - KEY_DATES["aground"]).dt.days
wreck_passes[["date", "stack_id", "pols", "days_since_aground"]]

- No pass falls on the explosion (8 June), when the tanker was still at sea

- The grounding is bracketed by two passes: `056_A` on **Jun 18**, five days before, and `150_D` on **Jun 24**, one day after

- From then on the two stacks alternate, so the wreck is seen about every 6 days, apart from the gap from Jul 27 to Aug 10

<br/>

#### 3.2 The tanker on the rocks

A ship is one of the brightest things a radar can see: flat steel walls and the deck make corners that send the pulse straight back ([SEOS, oil on radar](https://seos-project.eu/marinepollution/marinepollution-c02-s02-p03.html)). At 274 m long, the tanker is about 14 pixels at 20 m (frequency A, `150_D`), and about 3 at 80 m (frequency B, the VV-only `056_A_010` frame)

Read a small window around Jazirat Al Qibliyyah on every pass that covers it, to see when a bright spot appears at the southwest tip

In [ ]:
# window around the island, about 9 x 7 km
WRECK_BOX = (56.29, 17.47, 56.38, 17.53)

# function to read a small window of one layer around the wreck, in dB
def read_chip(url, bbox=WRECK_BOX):
    """
    Read HH (or VV when there's no HH) in dB over a small box, from the finest frequency the file has

    Args:
        url: The .h5 download link
        bbox: Bounding box (west, south, east, north) in degrees

    Returns:
        (db, x, y, frequency), or None if the file doesn't reach the box
    """
    with open_h5(url) as f:
        grids = f["science/LSAR/GCOV/grids"]
        freq = "frequencyA" if "frequencyA" in grids else "frequencyB"
        grid = grids[freq]
        layer = "HHHH" if "HHHH" in grid else "VVVV"
        window = aoi_window(grid, bbox)
        if window is None:
            return None
        rows, cols = window
        power, mask = grid[layer][rows, cols], grid["mask"][rows, cols]
        x, y = grid["xCoordinates"][cols], grid["yCoordinates"][rows]
    valid = (mask >= 1) & (mask <= 5) & (power > 0)
    return np.where(valid, 10 * np.log10(np.where(valid, power, 1)), np.nan), x, y, freq

In [ ]:
from pathlib import Path

# results are saved, so a rerun doesn't read every file again
CACHE = Path("../data/case4/cache")
CACHE.mkdir(parents=True, exist_ok=True)

chips = {}
for p in wreck_passes.itertuples():
    path = CACHE / f"chip_{p.stack_id}_{p.date:%Y%m%d}.npz"
    # a saved chip with no valid pixels came from the wrong frame, read it again
    if path.exists() and not np.isfinite(np.load(path)["db"]).any():
        path.unlink()
    if not path.exists():
        # a pass can have two frames that both reach the window, keep the one with valid pixels
        piece_urls = pieces[(pieces.stack_id == p.stack_id) & (pieces.date == p.date)].url
        results = [r for r in map(read_chip, piece_urls) if r is not None]
        result = max(results, key=lambda r: np.isfinite(r[0]).sum(), default=None)
        if result is None or not np.isfinite(result[0]).any():
            continue
        np.savez(path, db=result[0], x=result[1], y=result[2], freq=result[3])
    saved = np.load(path)
    chips[(p.date, p.stack_id)] = (saved["db"], saved["x"], saved["y"], str(saved["freq"]))
len(chips)

In [ ]:
# every chip in date order, the yellow cross is the centre of the island
ncols = 5
nrows = int(np.ceil(len(chips) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.8 * nrows))
to_utm = Transformer.from_crs(4326, 32640, always_xy=True)
for ax, ((date, stack_id), (db, x, y, freq)) in zip(axes.flat, sorted(chips.items())):
    ax.imshow(db, extent=[x[0], x[-1], y[-1], y[0]], cmap="gray", vmin=-30, vmax=5)
    ax.plot(*to_utm.transform(wreck.x, wreck.y), "y+", markersize=10)
    ax.set_title(f"{date:%b %d}, {stack_id}, {'20' if freq == 'frequencyA' else '80'} m", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[len(chips):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

- In the 80 m `056_A` windows a dark patch appears east and north-east of the island from **Jul 12**, grows through Jul 24, Aug 17 and Aug 29, and by Sep 10 and Sep 22 surrounds the island: the slick, one pass after another

- A bright spot sits at the southwest tip on every pass, already faint on Jun 18 before the grounding, so at 80 m the tanker can't be told apart from the rocks and surf there. The 20 m `150_D` windows are the closer look

<br/>

#### 3.3 Finding the slick

Oil calms the sea, so a slick is darker than the water **around it**. The sea's own brightness changes across the image (with the wind and the radar's viewing angle), so each pixel is compared with its surroundings rather than with one fixed value:

- **Smoothing**: each pixel is averaged with its 5 x 5 neighbours (400 m), to cut speckle

- **Background**: the usual brightness of the sea within about 12 km of each pixel. The slicks here are several km wide, so a smaller area would mostly be slick itself

- **Contrast**: the pixel minus its background, in dB

- **Slick**: sea pixels at least `THRESHOLD` dB darker than their background, in patches of at least 2 km²

- **Calm water**: where the sea around a pixel is itself 3 dB darker than usual, there's too little wind to tell oil from calm water, so it's left out. A pass where that's a big share of the sea is left out entirely

- **Edges**: the 500 m next to the coast and the edges of the image are left out, where the smoothing would mix in land or empty pixels

The slicks here are only about **1 to 2 dB** darker than the sea around them on most passes, up to 4 to 6 dB on Sep 10 (checked on the Aug 11, Aug 17, Sep 4 and Sep 10 passes), so the threshold is 1.5 dB

Land is masked out first, using the same elevation tiles as the 3D view (AWS Terrain Tiles, [Terrain Tiles](https://registry.opendata.aws/terrain-tiles/)): anything above sea level is land

This uses the 80 m frequency B layer, the one every pass has, HH for `150_D` and VV for `056_A`

In [ ]:
import io
from PIL import Image

EPSG, RES = 32640, 80
to_utm = Transformer.from_crs(4326, EPSG, always_xy=True)
(bx0, by0), (bx1, by1) = to_utm.transform(west, south), to_utm.transform(east, north)
grid_x = np.arange(bx0, bx1, RES)
grid_y = np.arange(by1, by0, -RES)

# function to read every frame of one pass and place them on one 80 m grid over the box
def read_pass(stack_id, date):
    """
    Mosaic the frequency B HH (or VV) of all the frames of one pass onto a common grid

    Args:
        stack_id: e.g. "150_D"
        date: Date of the pass

    Returns:
        2D array in dB on grid_x, grid_y, nan where there's no valid data
    """
    out = np.full((len(grid_y), len(grid_x)), np.nan, np.float32)
    for p in pieces[(pieces.stack_id == stack_id) & (pieces.date == date)].itertuples():
        result = read_box(p.url, "HHHH") or read_box(p.url, "VVVV")
        if result is None:
            continue
        power, mask, x, y = result
        valid = (mask >= 1) & (mask <= 5) & (power > 0)
        db = np.where(valid, 10 * np.log10(np.where(valid, power, 1)), np.nan)
        cols = np.round((x - grid_x[0]) / RES).astype(int)
        rows = np.round((grid_y[0] - y) / RES).astype(int)
        keep_c, keep_r = (cols >= 0) & (cols < len(grid_x)), (rows >= 0) & (rows < len(grid_y))
        target = np.ix_(rows[keep_r], cols[keep_c])
        out[target] = np.where(np.isnan(db[np.ix_(keep_r, keep_c)]), out[target], db[np.ix_(keep_r, keep_c)])
    return out

In [ ]:
# every pass over the wreck, read once and saved
sea_db = {}
for p in wreck_passes.itertuples():
    path = CACHE / f"pass_{p.stack_id}_{p.date:%Y%m%d}.npy"
    if not path.exists():
        np.save(path, read_pass(p.stack_id, p.date))
    sea_db[(p.date, p.stack_id)] = np.load(path)
len(sea_db)

In [ ]:
TERRAIN_TILES = "https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png"

# function to sample terrain height at many points from the map tiles
def terrain_height(lon, lat, z=10):
    """
    Height in metres from AWS Terrain Tiles (terrarium encoding) at each longitude and latitude

    Args:
        lon, lat: Arrays of the same shape, in degrees
        z: Tile zoom level, 10 is about 150 m per pixel here

    Returns:
        Array of heights in metres, same shape as lon
    """
    n = 256 * 2 ** z
    px = (lon + 180) / 360 * n
    py = (1 - np.log(np.tan(np.radians(lat)) + 1 / np.cos(np.radians(lat))) / np.pi) / 2 * n
    tx, ty = (px // 256).astype(int), (py // 256).astype(int)
    height = np.full(lon.shape, np.nan)
    for x_tile, y_tile in set(zip(tx.ravel(), ty.ravel())):
        tile = requests.get(TERRAIN_TILES.format(z=z, x=x_tile, y=y_tile), timeout=60)
        rgb = np.asarray(Image.open(io.BytesIO(tile.content)).convert("RGB")).astype(float)
        here = (tx == x_tile) & (ty == y_tile)
        r, c = (py[here] % 256).astype(int), (px[here] % 256).astype(int)
        height[here] = rgb[r, c, 0] * 256 + rgb[r, c, 1] + rgb[r, c, 2] / 256 - 32768
    return height

# land mask on the 80 m grid: anything above sea level
X, Y = np.meshgrid(grid_x, grid_y)
lon_grid, lat_grid = Transformer.from_crs(EPSG, 4326, always_xy=True).transform(X, Y)
land = terrain_height(lon_grid, lat_grid) > 0
print(f"land: {land.mean():.0%} of the box")

In [ ]:
import warnings
from scipy import ndimage

# blocks that are all land give an empty-mean warning, they're left as nan on purpose
warnings.filterwarnings("ignore", message="Mean of empty slice")

THRESHOLD = 1.5    # dB darker than the surrounding sea
SMOOTH = 5         # pixels, 400 m at 80 m
BLOCK = 4          # the background is worked out on a 4 x coarser grid (320 m), much faster
BACKGROUND = 75    # coarse pixels, about 24 km across, so about 12 km either side
MIN_KM2 = 2.0
EDGE = 6           # pixels left out next to the coast and the edges of the image, about 500 m
LOW_WIND = 3       # dB: sea whose surroundings are this much darker than the stack's usual sea is calm water, not counted
PIXEL_KM2 = (RES / 1000) ** 2

# function to average an image over a window, leaving out pixels that aren't valid sea
def masked_mean(values, valid, size):
    total = ndimage.uniform_filter(np.where(valid, values, 0.0), size)
    count = ndimage.uniform_filter(valid.astype(float), size)
    return np.where(count > 0.2, total / np.maximum(count, 1e-6), np.nan)

# function to find the slick on one pass
def find_slick(db, usual):
    """
    Sea pixels darker than the sea around them, in patches big enough to be a slick

    Args:
        db: 2D array in dB on the common grid
        usual: The usual brightness of the sea for this stack, in dB

    Returns:
        (slick, contrast, sea): boolean slick mask, contrast in dB, and boolean mask of valid sea
    """
    # stay clear of the coast and the image edges, where smoothing mixes in land or nothing
    sea = ndimage.binary_erosion(~land & ~np.isnan(db), iterations=EDGE)
    smooth = masked_mean(db, sea, SMOOTH)

    # background on a coarser grid, then back to 80 m
    h, w = db.shape[0] // BLOCK * BLOCK, db.shape[1] // BLOCK * BLOCK
    coarse_sea = sea[:h, :w].reshape(h // BLOCK, BLOCK, w // BLOCK, BLOCK).mean(axis=(1, 3)) > 0.5
    coarse_db = np.nanmean(np.where(sea, db, np.nan)[:h, :w].reshape(h // BLOCK, BLOCK, w // BLOCK, BLOCK), axis=(1, 3))
    coarse_bg = masked_mean(np.nan_to_num(coarse_db), coarse_sea, BACKGROUND)
    background = np.full(db.shape, np.nan)
    background[:h, :w] = np.repeat(np.repeat(coarse_bg, BLOCK, axis=0), BLOCK, axis=1)

    # big calm areas (little wind) are dark too, whether there's oil or not, so they're left out
    calm = background < usual - LOW_WIND
    contrast = np.where(sea & ~calm, smooth - background, np.nan)
    dark = ndimage.binary_opening(np.nan_to_num(contrast) < -THRESHOLD, iterations=2)
    labels, n = ndimage.label(dark)
    sizes = ndimage.sum(dark, labels, range(1, n + 1)) * PIXEL_KM2
    slick = np.isin(labels, 1 + np.flatnonzero(sizes >= MIN_KM2))
    return slick, contrast, sea

In [ ]:
# slick on every pass: area, sea seen, and where the slick sits compared with the wreck
wx, wy = to_utm.transform(wreck.x, wreck.y)

# the usual brightness of the sea for each stack (HH and VV differ), the middle value over all its passes
usual = {stack_id: np.nanmedian([np.nanmedian(db[~land]) for (d, s), db in sea_db.items() if s == stack_id])
         for stack_id in {s for _, s in sea_db}}

rows = []
slicks = {}
for (date, stack_id), db in sorted(sea_db.items()):
    slick, contrast, sea = find_slick(db, usual[stack_id])
    # share of the sea too calm to judge, a pass that's mostly calm is left out
    calm_share = 1 - np.isfinite(contrast).sum() / max(sea.sum(), 1)
    if calm_share > 0.3:
        slick = np.zeros_like(slick)
    slicks[(date, stack_id)] = slick
    ys, xs = np.nonzero(slick)
    far = np.hypot(grid_x[xs] - wx, grid_y[ys] - wy).max() / 1000 if len(xs) else 0
    rows.append({"date": date, "stack_id": stack_id, "sea_seen_km2": round(sea.sum() * PIXEL_KM2), "calm_share": round(calm_share, 2),
                 "slick_km2": round(slick.sum() * PIXEL_KM2, 1), "farthest_from_wreck_km": round(far, 1),
                 "centre_east_km": round((grid_x[xs].mean() - wx) / 1000, 1) if len(xs) else np.nan,
                 "centre_north_km": round((grid_y[ys].mean() - wy) / 1000, 1) if len(xs) else np.nan})
slick_table = pd.DataFrame(rows)
slick_table

- No slick before mid-July apart from a few small patches, then **Jul 24** and **Aug 17** show slicks within 50 km of the wreck

- The slick in view is largest in **September** (Sep 4 to Sep 22), a few hundred km² of the box on each pass

- **Aug 11** shows the trails by eye (2.4) but they're only about 1 dB darker than the sea, below the threshold, so it comes out empty

- **Oct 10** is left out: half the sea is calm water after the monsoon winds drop (`calm_share`), too dark to tell oil apart

In [ ]:
# slick (red) over the radar image for every pass, the yellow cross is the wreck
ncols = 4
nrows = int(np.ceil(len(sea_db) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4 * nrows))
extent = [grid_x[0], grid_x[-1], grid_y[-1], grid_y[0]]
for ax, ((date, stack_id), db) in zip(axes.flat, sorted(sea_db.items())):
    ax.imshow(db, extent=extent, cmap="gray", vmin=-30, vmax=0)
    ax.imshow(np.where(slicks[(date, stack_id)], 1.0, np.nan), extent=extent, cmap="Reds", vmin=0, vmax=1)
    ax.plot(wx, wy, "y+", markersize=10)
    ax.set_title(f"{date:%b %d}, {stack_id}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[len(sea_db):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

The red follows the dark streaks seen by eye on most passes: north-east of the wreck in August, wider bands east and north-east in September. A few small patches along the frame edges and the coast are likely calm water rather than oil

In [ ]:
# slick area over time against the key dates and the reported sizes
reported = pd.DataFrame({"date": pd.to_datetime(["2026-08-10"]), "km2": [390], "source": ["Oman, 10 Aug"]})
fig, ax = plt.subplots(figsize=(10, 3.5))
for stack_id, group in slick_table.groupby("stack_id"):
    ax.plot(group.date, group.slick_km2, marker="o", label=f"slick in the box, {stack_id}")
ax.scatter(reported.date, reported.km2, color="black", marker="x", s=60, label="reported, whole slick")
ax.axvspan(*GAP, color="gray", alpha=0.15)
for label, date in KEY_DATES.items():
    ax.axvline(date, color="red", linestyle="--", linewidth=1)
ax.set_ylabel("km²")
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b %d"))
ax.legend()
plt.show()

The box only holds part of the slick (Oman's 390 km² on 10 August was the whole slick, out to the open sea), and the two stacks differ (VV shows slicks more strongly than HH), so the numbers are best read as a trend: little before mid-July, growing in August, largest in September

Section 3 shows the spill clearly in the regular NISAR passes. Next is section 4: turning these results into files the application can use

<br/>
<br/>

### 4. Data Processing

Turn the results into small files the application can load directly. Everything goes in `dataScience/data/case4/export/`, flat, ready to copy into `application/public/assets/geospatial/case4/`:

- **Overlays**: the radar before the spill, the slick at its biggest, and every place the slick reached over the summer, as PNGs lined up with the map

- **Places**: labels for the 3D view

- **Charts**: one small chart per stage

- **`manifest.json`**: the case study for the app, in the same structure as the `CaseStudy` type in `caseStudies.ts`

#### 4.1 Overlays on the map grid

The app drapes images over a Web Mercator map (the grid most web maps use), while the radar results are on the UTM grid (2.1). Each result is moved onto a Web Mercator grid over the box before saving, so it sits on the right stretch of sea

In [ ]:
import json

OUT = Path("../data/case4/export")
OUT.mkdir(parents=True, exist_ok=True)

# Web Mercator row of a latitude, as a share of the world's height
merc_y = lambda lat: (1 - np.log(np.tan(np.radians(lat)) + 1 / np.cos(np.radians(lat))) / np.pi) / 2
WIDTH = 1024
HEIGHT = round(WIDTH * (merc_y(south) - merc_y(north)) / ((east - west) / 360))

# longitude and latitude of every pixel of the map grid, then where each one falls on the 80 m UTM grid
map_lon = np.linspace(west, east, WIDTH)
map_lat = np.degrees(np.arctan(np.sinh(np.pi * (1 - 2 * np.linspace(merc_y(north), merc_y(south), HEIGHT)))))
MX, MY = to_utm.transform(*np.meshgrid(map_lon, map_lat))
map_cols = np.round((MX - grid_x[0]) / RES).astype(int)
map_rows = np.round((grid_y[0] - MY) / RES).astype(int)
on_grid = (map_cols >= 0) & (map_cols < len(grid_x)) & (map_rows >= 0) & (map_rows < len(grid_y))

# function to move an array from the 80 m UTM grid onto the map grid
def to_map_grid(data, fill=np.nan):
    """
    Resample a UTM array onto the Web Mercator map grid, nearest pixel

    Args:
        data: 2D array on grid_x, grid_y
        fill: Value outside the data

    Returns:
        2D array, HEIGHT x WIDTH
    """
    out = np.full((HEIGHT, WIDTH), fill, dtype=float)
    out[on_grid] = data[map_rows[on_grid], map_cols[on_grid]]
    return out

# function to save an RGBA array as a PNG in the export folder
def save_png(rgba, name):
    Image.fromarray(rgba.astype(np.uint8), "RGBA").save(OUT / f"{name}.png")
    return f"case4/{name}.png"

print(f"map grid: {WIDTH} x {HEIGHT} pixels")

In [ ]:
# the passes the stages use: the first one over the wreck (before it went aground), and the one with the most slick
before_key = min(sea_db)
event_row = slick_table.loc[slick_table.slick_km2.idxmax()]
event_key = (event_row.date, event_row.stack_id)

# radar in grey: -25 dB black (calm water, oil) to -5 dB white (rough sea, land), see-through where there's no data
def radar_rgba(db):
    level = np.nan_to_num(np.clip((db + 25) / 20, 0, 1)) * 255
    alpha = np.where(np.isnan(db), 0, 235)
    return np.dstack([level, level, level, alpha])

SLICK_RGB = (220, 40, 40)
# slick in red, everything else see-through
def slick_rgba(slick):
    on = slick > 0.5
    return np.dstack([np.full(on.shape, c) for c in SLICK_RGB] + [np.where(on, 230, 0)])

files = {
    "radar_before": save_png(radar_rgba(to_map_grid(sea_db[before_key])), "radar_before"),
    "radar_event": save_png(radar_rgba(to_map_grid(sea_db[event_key])), "radar_event"),
    "slick_event": save_png(slick_rgba(to_map_grid(slicks[event_key].astype(float), 0)), "slick_event"),
}

# one slick layer per pass, for stepping through the summer
pass_files = {key: save_png(slick_rgba(to_map_grid(s.astype(float), 0)), f"slick_{key[0]:%Y%m%d}_{key[1]}")
              for key, s in slicks.items()}
print(f"before: {before_key[0]:%b %d} {before_key[1]}, event: {event_key[0]:%b %d} {event_key[1]}, {event_row.slick_km2} km²")

In [ ]:
from matplotlib import colormaps

# how many passes saw a slick at each pixel, over the whole summer
seen_slick = np.sum([s for s in slicks.values()], axis=0)
footprint_km2 = (seen_slick > 0).sum() * PIXEL_KM2
most = int(seen_slick.max())

# yellow for one pass, dark red for the most, see-through where the slick never was
count_map = to_map_grid(seen_slick.astype(float), 0)
rgba = colormaps["YlOrRd"](np.clip(count_map / max(most, 1), 0.25, 1)) * 255
rgba[..., 3] = np.where(count_map > 0, 220, 0)
files["slick_footprint"] = save_png(rgba, "slick_footprint")
print(f"sea covered by a slick on at least one pass: {footprint_km2:.0f} km², at most {most} passes at one spot")

In [ ]:
# check the overlays line up: the slick footprint over the radar before the spill
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(np.asarray(Image.open(OUT / "radar_before.png")), extent=[west, east, south, north])
ax.imshow(np.asarray(Image.open(OUT / "slick_footprint.png")), extent=[west, east, south, north])
ax.plot(wreck.x, wreck.y, "c+", markersize=12)
ax.set_title("Where the slick was seen over the summer")
plt.show()

<br/>

#### 4.2 Places

Labels for the 3D view: the wreck first, then the islands and towns from 1.2

In [ ]:
# places from 1.2, the wreck first
place_rows = [{"name": "Caroline Bezengi (wreck)", "lon": wreck.x, "lat": wreck.y}]
place_rows += [{"name": r.name, "lon": r.lon, "lat": r.lat} for r in places.itertuples() if "wreck" not in r.name]

geojson = lambda features: {"type": "FeatureCollection", "features": features}
with open(OUT / "places.geojson", "w") as file:
    json.dump(geojson([{"type": "Feature", "geometry": {"type": "Point", "coordinates": [p["lon"], p["lat"]]},
                        "properties": {"name": p["name"]}} for p in place_rows]), file, indent=1)

# the places over the footprint
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(np.asarray(Image.open(OUT / "radar_before.png")), extent=[west, east, south, north])
ax.imshow(np.asarray(Image.open(OUT / "slick_footprint.png")), extent=[west, east, south, north])
ax.scatter([p["lon"] for p in place_rows], [p["lat"] for p in place_rows], color="white", s=15)
for p in place_rows:
    ax.annotate(p["name"], (p["lon"], p["lat"]), color="white", fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.show()

<br/>

#### 4.3 Legends and mini charts

The app draws the legend from the colours of each overlay, and shows a small chart next to the story:

- **Lead-up**: the passes over the wreck against the explosion and the grounding

- **Event**: slick area on every pass, against the size Oman reported

- **Takeaway**: how far from the wreck the slick reached on every pass

In [ ]:
legends = {
    "radar": {"gradient": [[0, 0, 0], [255, 255, 255]],
              "labels": {"lowEnd": "-25 dB: calm water or oil", "highEnd": "-5 dB: rough sea, land"}},
    "slick": {"type": "categories", "gradient": [list(SLICK_RGB)],
              "labels": {"lowEnd": "Slick", "highEnd": "Slick"},
              "categories": [{"color": list(SLICK_RGB), "label": f"Slick: sea at least {THRESHOLD} dB darker than around it"}]},
    "footprint": {"gradient": [[round(c * 255) for c in colormaps["YlOrRd"](v)[:3]] for v in np.linspace(0.25, 1, 4)],
                  "labels": {"lowEnd": "Slick on 1 pass", "highEnd": f"Slick on {most} passes"}},
}

# colours of the app, so the charts sit well on the story card
INK, MUTED, ACCENT, ALERT = "#e8ebf3", "#aab3c8", "#c6f36b", "#ff6b4a"

# function to give a mini chart the app's look and save it
def save_chart(fig, ax, name):
    """
    Style a small chart for the dark story card and save it as a transparent PNG

    Args:
        fig, ax: The matplotlib figure and axes
        name: File name without the extension

    Returns:
        Path of the file as the app sees it
    """
    ax.set_facecolor("none")
    ax.tick_params(colors=MUTED, labelsize=8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(MUTED)
    ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator())
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b"))
    fig.savefig(OUT / f"{name}.png", dpi=150, transparent=True, bbox_inches="tight")
    plt.show()
    return f"case4/{name}.png"

charts = {}

# lead-up: every pass over the wreck, with the explosion and the grounding
fig, ax = plt.subplots(figsize=(4, 1.4))
for i, (stack_id, group) in enumerate(wreck_passes.groupby("stack_id")):
    ax.scatter(group.date, [i] * len(group), color=[MUTED, ACCENT][i], s=14)
# the two dates are close, so one label sits left of its line and the other right
for label, side in (("explosion", "right"), ("aground", "left")):
    ax.axvline(KEY_DATES[label], color=ALERT, linewidth=1.2)
    ax.text(KEY_DATES[label], 1.5, f" {label} ", color=ALERT, fontsize=7, ha=side)
ax.set_yticks([0, 1], sorted(wreck_passes.stack_id.unique()))
ax.set_ylim(-0.6, 2)
charts["leadUp"] = save_chart(fig, ax, "lead_up")

# event: slick area on every pass, against Oman's figure for the whole slick
fig, ax = plt.subplots(figsize=(4, 2))
ax.plot(slick_table.date, slick_table.slick_km2, color=ACCENT, marker="o", markersize=3, label="slick in the box")
ax.scatter(event_row.date, event_row.slick_km2, color=ALERT, s=40, zorder=3)
ax.scatter(KEY_DATES["slick 390 km²"], 390, color=INK, marker="x", s=30, label="Oman, whole slick")
ax.set_ylabel("km²", color=MUTED, fontsize=8)
ax.legend(frameon=True, facecolor="#121a30", edgecolor="none", labelcolor=INK, fontsize=7, loc="upper left")
charts["event"] = save_chart(fig, ax, "event")

# takeaway: how far from the wreck the slick reached
fig, ax = plt.subplots(figsize=(4, 2))
ax.plot(slick_table.date, slick_table.farthest_from_wreck_km, color=ACCENT, marker="o", markersize=3)
ax.set_ylabel("km from the wreck", color=MUTED, fontsize=8)
charts["takeaway"] = save_chart(fig, ax, "takeaway")

<br/>

#### 4.4 Story and manifest

Three stages, each with a short non-technical write-up, the overlay to show, its legend and chart. The numbers come from the analysis above, the wording is a first draft to edit

`manifest.json` follows the `CaseStudy` type in `application/src/data/caseStudies.ts`, so it can go straight into the app. File paths are relative to `application/public/assets/geospatial/`

In [ ]:
# compass direction of a point east and north of the wreck
def direction(east_km, north_km):
    names = ["north", "north-east", "east", "south-east", "south", "south-west", "west", "north-west"]
    return names[int(((np.degrees(np.arctan2(east_km, north_km)) + 360 + 22.5) % 360) // 45)]

event_date = pd.Timestamp(event_row.date)
event_way = direction(event_row.centre_east_km, event_row.centre_north_km)
foot_e, foot_n = (grid_x[xs].mean() - wx) / 1000, (grid_y[ys].mean() - wy) / 1000
first_after = wreck_passes[wreck_passes.date > KEY_DATES["aground"]].iloc[0]

stages = [
    {"type": "leadUp",
     "headline": "A tanker on the rocks, with NISAR passing every few days",
     "metric": {"label": "first pass after the tanker went aground", "value": f"{first_after.days_since_aground} day{'s' if first_after.days_since_aground != 1 else ''}"},
     "summary": "An explosion disabled the Caroline Bezengi, carrying close to a million barrels of crude, on 8 June. Around 23 June it ended up "
                "on the rocks at the southwest tip of Jazirat Al Qibliyyah, in the Hallaniyat Islands. NISAR passed over the islands on "
                f"{before_key[0]:%-d %B} and again on {first_after.date:%-d %B}. On radar the sea is grey, the wind's small ripples sending part of the pulse back",
     "confidence": "Medium",
     "imagery": files["radar_before"],
     "legend": legends["radar"],
     "chart": charts["leadUp"]},
    {"type": "event",
     "headline": "The slick shows up as calm, dark water",
     "metric": {"label": f"of slick in view on {event_date:%-d %B}", "value": f"{event_row.slick_km2:.0f} km²"},
     "summary": "Oil smooths the ripples on the sea, so less of the radar pulse comes back and the slick looks darker than the water around it. "
                f"On {event_date:%-d %B}, dark streaks run {event_way} from the wreck for up to {event_row.farthest_from_wreck_km:.0f} km. "
                "Oman put the whole slick at about 390 km² on 10 August, and on 13 August oil reached the beaches at Ras Madrakah",
     "confidence": "Medium",
     "imagery": files["slick_event"],
     "legend": legends["slick"],
     "chart": charts["event"]},
    {"type": "takeaway",
     "headline": "Regular passes followed the spill all summer",
     "metric": {"label": "passes over the wreck, June to October", "value": str(len(wreck_passes))},
     "summary": "NISAR never switched this area to emergency processing, yet its regular passes saw the wreck about every 6 days. "
                f"Put together, they show where the oil went: {footprint_km2:.0f} km² of sea had a slick on at least one pass, "
                f"mostly {direction(foot_e, foot_n)} of the islands. Radar sees the sea day or night and through cloud, so it can follow "
                "a spill when the weather keeps boats and planes away",
     "confidence": "Medium",
     "imagery": files["slick_footprint"],
     "legend": legends["footprint"],
     "chart": charts["takeaway"],
     "actions": ["Map slicks on every regular NISAR pass near a stricken ship, without waiting for an emergency request",
                 "Use the drift direction to warn the coast and protected areas downwind early"]},
]

for s in stages:
    print(f"{s['type']}: {s['headline']}\n  {s['metric']['value']} {s['metric']['label']}\n  {s['summary']}\n")

In [ ]:
# one case study, in the same structure as the CaseStudy type in caseStudies.ts
manifest = {
    "case": "case4",
    "title": "Caroline Bezengi oil spill, Hallaniyat Islands, Oman (June to September 2026)",
    "location": "Hallaniyat Islands, Oman",
    "event_date": "2026-06-23",
    "story": "On 8 June 2026 an explosion disabled the Caroline Bezengi, a sanctioned tanker carrying close to a million barrels of Russian crude. "
             "Around 23 June it ended up on the rocks at Jazirat Al Qibliyyah, one of the Hallaniyat Islands off Oman, and leaked through the monsoon. "
             "By 10 August Oman put the slick at about 390 km², and on 13 August oil reached the beaches at Ras Madrakah",
    "method": "Oil smooths the small wind ripples on the sea, so it sends less of the radar pulse back and looks darker than the water around it. "
              f"On every NISAR pass over the wreck, sea pixels at least {THRESHOLD} dB darker than the sea within about 12 km, in patches of "
              f"{MIN_KM2:.0f} km² or more, are marked as slick. Frequency B images at 80 m, HH or VV",
    "limitations": "Calm patches of sea with little wind also look dark, so some marked areas may not be oil. At L-band the slicks here are "
                   "only 1 to 2 dB darker than the sea on most passes. The box covers the wreck and the coast up to Ras Madrakah, not the "
                   "whole slick, and passes are about 6 days apart, with no data from 27 July to 10 August",
    "coordinates": list(AOI),
    "image_crs": "EPSG:3857",
    "satellite": "NISAR (NASA-ISRO), L-band, track 150 descending (HH) and track 56 ascending (VV)",
    "sources": [
        "NISAR L2 GCOV, Provisional (NISAR_L2_GCOV_PROVISIONAL_V1), NASA Earthdata / ASF",
        "NISAR Data User Guide: https://nisar-docs.asf.alaska.edu/",
        "Maritime Executive, grounding and timeline: https://maritime-executive.com/article/stricken-tanker-caroline-bezengi-continues-to-leak-oil-off-oman",
        "Reuters via gCaptain, slick size: https://gcaptain.com/oil-spill-off-oman-from-tanker-under-sanctions-against-russia-spreads-over-huge-area/",
        "Al Jazeera, oil on the coast: https://www.aljazeera.com/news/2026/8/13/oman-says-oil-spill-from-stricken-tanker-has-reached-its-coast",
        "OpenStreetMap and Wikipedia, place locations",
        "Analysis: https://github.com/danesheppard/PanoptiCrew-NASA-Hackathon-2026 (dataScience/notebooks/case4.ipynb)",
    ],
    "key_dates": {"before": f"{before_key[0]:%Y-%m-%d}", "event": f"{event_date:%Y-%m-%d}", "after": f"{max(sea_db)[0]:%Y-%m-%d}"},
    "scene": {"places": "case4/places.geojson"},
    "layers": [
        {"name": f"NISAR radar, {before_key[0]:%b %d}, before the spill", "file": files["radar_before"], "legend": legends["radar"]},
        {"name": f"NISAR radar, {event_date:%b %d}", "file": files["radar_event"], "legend": legends["radar"]},
        {"name": f"Slick, {event_date:%b %d}", "file": files["slick_event"], "legend": legends["slick"]},
        {"name": "Slick footprint, June to October", "file": files["slick_footprint"], "legend": legends["footprint"]},
    ],
    "pairs": [{"date": f"{r.date:%Y-%m-%d}", "stack": r.stack_id, "file": pass_files[(r.date, r.stack_id)],
               "slick_km2": float(r.slick_km2), "farthest_from_wreck_km": float(r.farthest_from_wreck_km)}
              for r in slick_table.itertuples()],
    "stages": stages,
}
with open(OUT / "manifest.json", "w") as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

# a short readme next to the files
(OUT / "README.md").write_text(f"""# Case 4 export: Caroline Bezengi oil spill seen by NISAR

Produced by `notebooks/case4.ipynb` (section 4). Copy these files to `application/public/assets/geospatial/case4/`

Every image covers the box in `manifest.json` → `coordinates`, in Web Mercator (EPSG:3857), {WIDTH} x {HEIGHT} px. Transparent pixels mean nothing to show

| File | What it is |
|---|---|
| `radar_before.png`, `radar_event.png` | NISAR radar in grey, before the spill and on the pass with the most slick |
| `slick_event.png` | slick on that pass, red |
| `slick_footprint.png` | how many passes saw a slick at each spot |
| `slick_<date>_<stack>.png` | slick on each pass, for a timeline |
| `places.geojson` | labels for the 3D view |
| `lead_up.png`, `event.png`, `takeaway.png` | mini chart for each stage |
| `manifest.json` | the case study for the app, same structure as `CaseStudy` in `caseStudies.ts` |
""")

pd.DataFrame([(p.name, round(p.stat().st_size / 1e3, 1)) for p in sorted(OUT.iterdir())], columns=["file", "kb"])

<br/>
<br/>